# Preprocessing Teks Ulasan E-commerce Berbahasa Indonesia

**Nama:** Muhammad Hafidz   
**NIM:** 24146028

Notebook ini melakukan preprocessing teks pada dataset ulasan produk e-commerce berbahasa Indonesia:
[`dipawidia/ecommerce-product-reviews-sentiment`](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment).

Tujuan notebook:
1. Memuat seluruh split dataset dan memeriksa struktur datanya secara nyata (tanpa menebak nama kolom).
2. Menampilkan informasi kualitas data: jumlah, tipe data, missing value, duplikat, dan distribusi label.
3. Menjalankan preprocessing bertahap: case folding, cleaning, tokenisasi, stopword removal, dan stemming.
4. Menampilkan contoh sebelum/sesudah serta mengukur dampak preprocessing terhadap panjang teks dan ukuran vocabulary.

## 1. Setup

Sel ini menyiapkan library, konfigurasi tampilan, serta folder keluaran (`data/`, `figures/`, `outputs/`). Seed dibuat tetap agar pemilihan contoh acak dapat direproduksi.

In [1]:
import os, re, json, time, random, warnings, datetime
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
FIG_DIR = BASE_DIR / "figures"
OUT_DIR = BASE_DIR / "outputs"
for folder in (DATA_DIR, FIG_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ID = "dipawidia/ecommerce-product-reviews-sentiment"
print("Working directory :", BASE_DIR)
print("Dataset           :", DATASET_ID)

Working directory : d:\NLP\preprocessing-ulasan-ecommerce-id
Dataset           : dipawidia/ecommerce-product-reviews-sentiment


## 2. Memuat Dataset

Dataset dimuat dengan `datasets.load_dataset`. Jika cara tersebut gagal (misalnya masalah jaringan atau versi library), notebook otomatis beralih ke `huggingface_hub.snapshot_download` lalu membaca file Parquet/CSV hasil unduhan.

Semua split yang tersedia digabung menjadi satu DataFrame. Kolom `split_asal` ditambahkan agar asal baris tetap terlacak. Nama kolom **tidak ditebak** pada tahap ini; kolom teks dan label dideteksi pada sel berikutnya setelah struktur data diperiksa.

In [2]:
splits = {}
load_source = None

try:
    from datasets import load_dataset
    ds = load_dataset(DATASET_ID)
    for split_name, split_ds in ds.items():
        splits[split_name] = split_ds.to_pandas()
    load_source = "datasets.load_dataset"
    print("Berhasil memuat dataset dengan datasets.load_dataset().")
except Exception as exc:
    print("datasets.load_dataset gagal:", repr(exc))
    print("Mencoba fallback melalui huggingface_hub.snapshot_download ...")
    from huggingface_hub import snapshot_download
    local_dir = snapshot_download(repo_id=DATASET_ID, repo_type="dataset")
    print("Lokasi snapshot:", local_dir)
    files = sorted(Path(local_dir).rglob("*.parquet")) or sorted(Path(local_dir).rglob("*.csv"))
    if not files:
        raise FileNotFoundError("Tidak ditemukan berkas Parquet/CSV pada snapshot dataset.")
    for file_path in files:
        part = pd.read_parquet(file_path) if file_path.suffix == ".parquet" else pd.read_csv(file_path)
        splits[file_path.stem] = part
    load_source = "huggingface_hub.snapshot_download"

frames = []
for split_name, part in splits.items():
    part = part.copy()
    if "split_asal" not in part.columns:
        part["split_asal"] = split_name
    frames.append(part)
df = pd.concat(frames, ignore_index=True)

print("Sumber pemuatan :", load_source)
print("Split tersedia  :", {name: len(part) for name, part in splits.items()})
print("Total baris     :", len(df))
print("Nama kolom      :", list(df.columns))
display(df.head())

Berhasil memuat dataset dengan datasets.load_dataset().
Sumber pemuatan : datasets.load_dataset
Split tersedia  : {'train': 11606}
Total baris     : 11606
Nama kolom      : ['review', 'translate', 'sentimen', 'split_asal']


,review,translate,sentimen,split_asal
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,"sound and mic satisfied. I only ordered the one that was on and I made sure that it clicked again, but the one that came didn't turn on",0,train
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1,train
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay attention again",0,train
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nit...,"product quality is good. affordable prices and very fast delivery. I've already ordered here, it's okay, but unfortunately, if you make ...",0,train
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit","overall is good, the buttons all work, but the packaging is lacking because it's dented and torn a little",0,train


## 3. Deteksi Otomatis Kolom Teks dan Label

Nama kolom tidak diasumsikan. Setiap kolom dihitung tipe data, jumlah nilai unik, dan rata-rata panjang teksnya. Kolom teks dipilih berdasarkan kombinasi kata kunci nama kolom (mis. `review`, `ulasan`, `text`) dan panjang teks rata-rata. Kolom label dipilih dari kolom non-teks dengan jumlah kelas sedikit (2–10), dengan preferensi nama yang mengandung `sentimen`/`label`.

Setelah kolom terdeteksi, isi tiap label ditampilkan untuk memverifikasi makna label sebelum dipetakan ke nama yang mudah dibaca.

In [3]:
def is_text_like(series):
    return pd.api.types.is_string_dtype(series) or series.dtype == object

col_stats = []
for col in df.columns:
    series = df[col]
    row = {
        "kolom": col,
        "tipe": str(series.dtype),
        "n_unique": int(series.nunique(dropna=True)),
        "rata2_panjang": round(float(series.astype(str).str.len().mean()), 2) if is_text_like(series) else np.nan,
    }
    col_stats.append(row)
col_stats_df = pd.DataFrame(col_stats)
display(col_stats_df)

TEXT_KEYWORDS = ("review", "ulasan", "komentar", "teks", "text", "isi", "content")
NON_TEXT_KEYWORDS = ("translate", "translation", "terjemah", "label", "sentimen", "sentiment", "split")

text_candidates = []
for col in df.columns:
    if not is_text_like(df[col]) or df[col].nunique(dropna=True) < 10:
        continue
    name = col.lower()
    score = float(df[col].astype(str).str.len().mean())
    if any(key in name for key in TEXT_KEYWORDS):
        score += 10000
    if any(key in name for key in NON_TEXT_KEYWORDS):
        score -= 10000
    text_candidates.append((score, col))

if not text_candidates:
    raise ValueError("Tidak ada kolom teks yang memenuhi kriteria.")

text_candidates.sort(reverse=True)
TEXT_COL = text_candidates[0][1]
print("Kandidat kolom teks (skor, kolom):")
for score, col in text_candidates:
    print(f"  {col:20s} skor={score:.2f}")
print("Kolom teks terpilih :", TEXT_COL)

LABEL_KEYWORDS = ("sentimen", "sentiment", "label", "kelas", "class", "target")
label_candidates = []
for col in df.columns:
    if col == TEXT_COL:
        continue
    n_unique = df[col].nunique(dropna=True)
    if 2 <= n_unique <= 10:
        name = col.lower()
        score = 0.0
        if any(key in name for key in LABEL_KEYWORDS):
            score += 1000
        if pd.api.types.is_numeric_dtype(df[col]) or isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == bool:
            score += 10
        score -= n_unique
        label_candidates.append((score, col, n_unique))

if not label_candidates:
    raise ValueError("Tidak ada kolom label kandidat dengan 2-10 kelas.")

label_candidates.sort(reverse=True)
LABEL_COL = label_candidates[0][1]
print("Kandidat kolom label (skor, kolom, jumlah kelas):")
for score, col, n_unique in label_candidates:
    print(f"  {col:20s} skor={score:.2f} kelas={n_unique}")
print("Kolom label terpilih:", LABEL_COL)

print(f"\nContoh teks untuk setiap nilai kolom '{LABEL_COL}':")
for value in sorted(df[LABEL_COL].dropna().unique()):
    print(f"\nLabel {value}:")
    for text in df.loc[df[LABEL_COL] == value, TEXT_COL].head(3):
        print("  -", str(text)[:160])

,kolom,tipe,n_unique,rata2_panjang
0,review,str,11599,103.57
1,translate,str,11584,109.75
2,sentimen,int64,2,NaN
3,split_asal,str,1,5.00


Kandidat kolom teks (skor, kolom):
  review               skor=10103.57
  translate            skor=-9890.25
Kolom teks terpilih : review
Kandidat kolom label (skor, kolom, jumlah kelas):
  sentimen             skor=1008.00 kelas=2
Kolom label terpilih: sentimen

Contoh teks untuk setiap nilai kolom 'sentimen':

Label 0:
  - Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala
  - kondisi dus robek2, tolong di perhatikan lagi
  - Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan ata

Label 1:
  - Good. Mudah-mudahan awet. Makasih 
  - Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.
  - terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka


Pemetaan label ditentukan setelah memeriksa contoh di atas. Pada dataset ini label `0` berisi keluhan/ulasan negatif dan label `1` berisi pujian/ulasan positif. Pemetaan hanya diterapkan jika nilai label tepat `{0, 1}`; selain itu label dibiarkan apa adanya.

In [4]:
unique_labels = sorted(df[LABEL_COL].dropna().unique().tolist())
if set(unique_labels) == {0, 1}:
    LABEL_MAP = {0: "Negatif", 1: "Positif"}
else:
    LABEL_MAP = {value: str(value) for value in unique_labels}

df["label_sentimen"] = df[LABEL_COL].map(LABEL_MAP)
print("Pemetaan label:", LABEL_MAP)
display(df[[TEXT_COL, LABEL_COL, "label_sentimen"]].head())

Pemetaan label: {0: 'Negatif', 1: 'Positif'}


,review,sentimen,label_sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,0,Negatif
1,Good. Mudah-mudahan awet. Makasih,1,Positif
2,"kondisi dus robek2, tolong di perhatikan lagi",0,Negatif
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nit...,0,Negatif
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",0,Negatif


## 4. Informasi dan Kualitas Dataset

Bagian ini melaporkan jumlah data per split dan total, tipe data setiap kolom, missing value, duplikat, serta distribusi label sentimen. Distribusi disajikan sebagai tabel (jumlah dan persentase) dan diagram batang yang disimpan ke `figures/distribusi_label.png`.

In [5]:
split_counts = df["split_asal"].value_counts().sort_index()
split_table = split_counts.rename("jumlah").to_frame()
split_table["persentase"] = (split_table["jumlah"] / len(df) * 100).round(2)
print("Jumlah data per split:")
display(split_table)
print("Total data:", len(df))

tipe_df = pd.DataFrame({
    "kolom": df.columns,
    "tipe_data": [str(dtype) for dtype in df.dtypes],
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
})
display(tipe_df)

n_missing = int(df.isna().sum().sum())
n_dup = int(df.duplicated().sum())
print("Total missing value :", n_missing)
print("Jumlah baris duplikat:", n_dup)

dist = df["label_sentimen"].value_counts().rename_axis("label").reset_index(name="jumlah")
dist["persentase"] = (dist["jumlah"] / len(df) * 100).round(2)
display(dist)
dist.to_csv(OUT_DIR / "distribusi_label.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
colors = ["#d9534f", "#5cb85c", "#5bc0de", "#f0ad4e"][:len(dist)]
bars = ax.bar(dist["label"].astype(str), dist["jumlah"], color=colors)
for bar, pct in zip(bars, dist["persentase"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{int(bar.get_height())}\n({pct}%)", ha="center", va="bottom", fontsize=9)
ax.set_title("Distribusi Label Sentimen")
ax.set_xlabel("Label sentimen")
ax.set_ylabel("Jumlah ulasan")
ax.set_ylim(0, dist["jumlah"].max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(FIG_DIR / "distribusi_label.png", dpi=160)
plt.show()

info_dataset = {
    "dataset_id": DATASET_ID,
    "sumber_pemuatan": load_source,
    "jumlah_per_split": {str(k): int(v) for k, v in split_counts.items()},
    "total_baris": int(len(df)),
    "kolom": [str(c) for c in df.columns],
    "tipe_data": {str(c): str(t) for c, t in df.dtypes.items()},
    "missing_per_kolom": {str(c): int(v) for c, v in df.isna().sum().items()},
    "total_missing": n_missing,
    "duplikat": n_dup,
    "kolom_teks": TEXT_COL,
    "kolom_label": LABEL_COL,
    "label_map": {str(k): str(v) for k, v in LABEL_MAP.items()},
    "distribusi_label": dist.to_dict(orient="records"),
}
with open(OUT_DIR / "info_dataset.json", "w", encoding="utf-8") as handle:
    json.dump(info_dataset, handle, ensure_ascii=False, indent=2)
print("Info dataset disimpan ke:", OUT_DIR / "info_dataset.json")

Jumlah data per split:


,jumlah,persentase
split_asal,,
train,11606,100.0


Total data: 11606


,kolom,tipe_data,non_null,missing
0,review,str,11606,0
1,translate,str,11606,0
2,sentimen,int64,11606,0
3,split_asal,str,11606,0
4,label_sentimen,str,11606,0


Total missing value : 0
Jumlah baris duplikat: 4


,label,jumlah,persentase
0,Negatif,5803,50.0
1,Positif,5803,50.0


Info dataset disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\outputs\info_dataset.json


## 5. Preprocessing

Preprocessing dilakukan bertahap, masing-masing dalam fungsi terpisah:

1. **Case folding** — mengubah seluruh teks menjadi huruf kecil agar kata seperti `Bagus` dan `bagus` dianggap sama.
2. **Cleaning** — menghapus URL, entitas HTML, angka, tanda baca, emoji/karakter non-alfabet, lalu merapikan spasi berlebih.
3. **Tokenisasi** — memecah teks bersih menjadi token kata.
4. **Stopword removal** — membuang kata umum berbahasa Indonesia. Daftar diambil dari Sastrawi dan (jika tersedia) NLTK. Kata negasi (`tidak`, `bukan`, `kurang`, `belum`, `jangan`, dan variannya) **sengaja dipertahankan** agar polaritas sentimen tidak berubah.
5. **Stemming** — mereduksi kata berimbuhan ke bentuk dasar dengan Sastrawi. Karena stemming relatif lambat, hasil per kata unik disimpan dalam cache `dict`, lalu dipetakan kembali ke seluruh token.

Hasil setiap tahap disimpan pada kolom tersendiri agar dapat diaudit.

In [6]:
import nltk
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

NEGATIONS = {"tidak", "bukan", "jangan", "belum", "kurang", "tanpa", "gak", "nggak", "ga", "tak", "enggak", "ndak"}

stopword_factory = StopWordRemoverFactory()
sw_sastrawi = set(stopword_factory.get_stop_words())

sw_nltk = set()
try:
    try:
        from nltk.corpus import stopwords as nltk_stopwords
        sw_nltk = set(nltk_stopwords.words("indonesian"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords as nltk_stopwords
        sw_nltk = set(nltk_stopwords.words("indonesian"))
except Exception as exc:
    print("Stopword NLTK tidak tersedia, memakai Sastrawi saja:", repr(exc))

ALL_STOPWORDS = sw_sastrawi | sw_nltk
STOPWORDS = ALL_STOPWORDS - NEGATIONS

print("Jumlah stopword Sastrawi :", len(sw_sastrawi))
print("Jumlah stopword NLTK     :", len(sw_nltk))
print("Stopword dipakai         :", len(STOPWORDS))
print("Negasi dipertahankan     :", sorted(NEGATIONS & ALL_STOPWORDS))

stemmer = StemmerFactory().create_stemmer()
stem_cache = {}

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
HTML_PATTERN = re.compile(r"&[a-zA-Z]+;|&#\d+;")
NON_ALPHA_PATTERN = re.compile(r"[^a-z\s]")
MULTISPACE_PATTERN = re.compile(r"\s+")

def case_folding(text):
    if pd.isna(text):
        return ""
    return str(text).lower()

def cleaning(text):
    text = URL_PATTERN.sub(" ", text)
    text = HTML_PATTERN.sub(" ", text)
    text = NON_ALPHA_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text)
    return text.strip()

def tokenize(text):
    return text.split()

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOPWORDS]

def stem_token(token):
    if token not in stem_cache:
        stem_cache[token] = stemmer.stem(token)
    return stem_cache[token]

def stemming(tokens):
    return [stem_token(token) for token in tokens]

def preprocess_pipeline(text):
    folded = case_folding(text)
    cleaned = cleaning(folded)
    tokens = tokenize(cleaned)
    without_stopwords = remove_stopwords(tokens)
    stemmed = stemming(without_stopwords)
    return folded, cleaned, tokens, without_stopwords, stemmed, " ".join(stemmed)

contoh_uji = "Produk BAGUS banget!!! Pengiriman cepat 2 hari, tapi sayang tidak ada bonus :("
for nama, nilai in zip(
    ["Case folding", "Cleaning", "Tokenisasi", "Stopword removal", "Stemming", "Final"],
    preprocess_pipeline(contoh_uji),
):
    print(f"{nama:18s}: {nilai}")

Jumlah stopword Sastrawi : 123
Jumlah stopword NLTK     : 757
Stopword dipakai         : 765
Negasi dipertahankan     : ['belum', 'bukan', 'enggak', 'jangan', 'kurang', 'nggak', 'tak', 'tanpa', 'tidak']
Case folding      : produk bagus banget!!! pengiriman cepat 2 hari, tapi sayang tidak ada bonus :(
Cleaning          : produk bagus banget pengiriman cepat hari tapi sayang tidak ada bonus
Tokenisasi        : ['produk', 'bagus', 'banget', 'pengiriman', 'cepat', 'hari', 'tapi', 'sayang', 'tidak', 'ada', 'bonus']
Stopword removal  : ['produk', 'bagus', 'banget', 'pengiriman', 'cepat', 'sayang', 'tidak', 'bonus']
Stemming          : ['produk', 'bagus', 'banget', 'kirim', 'cepat', 'sayang', 'tidak', 'bonus']
Final             : produk bagus banget kirim cepat sayang tidak bonus


Sel berikut menjalankan pipeline pada seluruh baris. Kemajuan dicetak setiap 2.000 baris. Waktu eksekusi dan jumlah kata unik yang di-stem juga dilaporkan.

In [7]:
start_time = time.time()
case_col, clean_col, token_col, sw_col, stem_col = [], [], [], [], []

for index, text in enumerate(df[TEXT_COL].tolist()):
    folded, cleaned, tokens, without_stopwords, stemmed, _ = preprocess_pipeline(text)
    case_col.append(folded)
    clean_col.append(cleaned)
    token_col.append(tokens)
    sw_col.append(without_stopwords)
    stem_col.append(stemmed)
    if (index + 1) % 2000 == 0:
        print(f"  diproses {index + 1}/{len(df)} baris ...")

df["case_folding"] = case_col
df["cleaning"] = clean_col
df["tokenisasi"] = token_col
df["stopword_removal"] = sw_col
df["stemming"] = stem_col
df["teks_final"] = [" ".join(tokens) for tokens in stem_col]

elapsed = time.time() - start_time
print(f"Selesai dalam {elapsed:.2f} detik")
print("Jumlah kata unik yang di-stem (cache):", len(stem_cache))

  diproses 2000/11606 baris ...


  diproses 4000/11606 baris ...


  diproses 6000/11606 baris ...


  diproses 8000/11606 baris ...


  diproses 10000/11606 baris ...


Selesai dalam 1311.44 detik
Jumlah kata unik yang di-stem (cache): 12082


### Penanganan Teks Kosong

Sebagian ulasan dapat menjadi kosong setelah preprocessing (misalnya hanya berisi angka, emoji, atau stopword). Baris seperti ini **tidak dihapus**, tetapi dihitung dan ditandai pada kolom `teks_kosong` agar transparan.

In [8]:
empty_mask = df["teks_final"].str.strip().eq("") | df["teks_final"].isna()
n_empty = int(empty_mask.sum())
df["teks_kosong"] = empty_mask

print(f"Jumlah teks kosong setelah preprocessing: {n_empty} dari {len(df)} baris ({n_empty / len(df) * 100:.2f}%)")
if n_empty:
    display(df.loc[empty_mask, [TEXT_COL, "case_folding", "cleaning", "teks_final"]].head(10))
else:
    print("Tidak ada teks yang kosong setelah preprocessing.")

Jumlah teks kosong setelah preprocessing: 16 dari 11606 baris (0.14%)


,review,case_folding,cleaning,teks_final
1489,lama,lama,lama,
3193,",,,,,,👍",",,,,,,👍",,
3404,\,\,,
5085,😍🖒😍🖒😍🖒😍🖒🖒😍🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒😍🖒🖒🖒😍🖒😍🖒🖒😍🖒😍,😍🖒😍🖒😍🖒😍🖒🖒😍🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒🖒😍🖒😍🖒😍🖒🖒🖒😍🖒😍🖒🖒😍🖒😍,,
5358,"Sedikit lama,,,","sedikit lama,,,",sedikit lama,
6272,ok,ok,ok,
6629,Aku kira panjang,aku kira panjang,aku kira panjang,
6880,bisa digunakan,bisa digunakan,bisa digunakan,
7648,ok ok ok ok ok ok ok ok ok,ok ok ok ok ok ok ok ok ok,ok ok ok ok ok ok ok ok ok,
8155,Lama sekali…………,lama sekali…………,lama sekali,


## 6. Contoh Sebelum dan Sesudah Preprocessing

Sepuluh baris dipilih acak dengan `random_state=42`. Tabel menampilkan teks asli, hasil case folding, cleaning, tokenisasi, stopword removal, dan stemming (hasil akhir). Tabel lengkap juga disimpan ke `outputs/contoh_10.csv`.

In [9]:
contoh = df.sample(n=10, random_state=RANDOM_STATE).copy()
contoh_tabel = pd.DataFrame({
    "No": range(1, 11),
    "Teks Asli": contoh[TEXT_COL].astype(str).values,
    "Case Folding": contoh["case_folding"].values,
    "Cleaning": contoh["cleaning"].values,
    "Tokenisasi": [" ".join(tokens) for tokens in contoh["tokenisasi"]],
    "Stopword Removal": [" ".join(tokens) for tokens in contoh["stopword_removal"]],
    "Stemming (Final)": contoh["teks_final"].values,
})

pd.set_option("display.max_colwidth", 220)
display(contoh_tabel)
contoh_tabel.to_csv(OUT_DIR / "contoh_10.csv", index=False)
print("Tabel 10 contoh disimpan ke:", OUT_DIR / "contoh_10.csv")

,No,Teks Asli,Case Folding,Cleaning,Tokenisasi,Stopword Removal,Stemming (Final)
0,1,Barangnya original 👍. Bisa muat banyak. Gampang dipakai. Penjualnya kurang ramah,barangnya original 👍. bisa muat banyak. gampang dipakai. penjualnya kurang ramah,barangnya original bisa muat banyak gampang dipakai penjualnya kurang ramah,barangnya original bisa muat banyak gampang dipakai penjualnya kurang ramah,barangnya original muat gampang dipakai penjualnya kurang ramah,barang original muat gampang pakai jual kurang ramah
1,2,Barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang pengirimannya cepat,barang kirim cepat
2,3,"realpict, packing oke","realpict, packing oke",realpict packing oke,realpict packing oke,realpict packing oke,realpict packing oke
3,4,Cuman bisa komen 👍👍👍,cuman bisa komen 👍👍👍,cuman bisa komen,cuman bisa komen,cuman komen,cuman komen
4,5,"mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.Desain elegan 😍. Resolusi gambar tajam 👍.","mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.desain elegan 😍. resolusi gambar tajam 👍.",mantap banget pengiriman cepat pesan sekarang besok sudah ditangan desain elegan resolusi gambar tajam,mantap banget pengiriman cepat pesan sekarang besok sudah ditangan desain elegan resolusi gambar tajam,mantap banget pengiriman cepat pesan besok ditangan desain elegan resolusi gambar tajam,mantap banget kirim cepat pesan besok tang desain elegan resolusi gambar tajam
5,6,"awalnya dikrm hanya unit saja, setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah habis..jadi tidak buat pelanggan kecewa","awalnya dikrm hanya unit saja, setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah habis..jadi tidak buat pelanggan kecewa",awalnya dikrm hanya unit saja setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah habis jadi tidak buat pelanggan kecewa,awalnya dikrm hanya unit saja setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah habis jadi tidak buat pelanggan kecewa,dikrm unit komplain dikrm info bonus bundling diupdate kalo tumbler dah habis tidak pelanggan kecewa,dikrm unit komplain dikrm info bonus bundling diupdate kalo tumbler dah habis tidak langgan kecewa
6,7,"menurut infonya free Mola TV tp ternyata perlu bayar, setelah diskusi panjang akhirnya diketahui kalau OS nya tidak diupdate...pembelian belum ada 7 hari sejak TV di terima...","menurut infonya free mola tv tp ternyata perlu bayar, setelah diskusi panjang akhirnya diketahui kalau os nya tidak diupdate...pembelian belum ada 7 hari sejak tv di terima...",menurut infonya free mola tv tp ternyata perlu bayar setelah diskusi panjang akhirnya diketahui kalau os nya tidak diupdate pembelian belum ada hari sejak tv di terima,menurut infonya free mola tv tp ternyata perlu bayar setelah diskusi panjang akhirnya diketahui kalau os nya tidak diupdate pembelian belum ada hari sejak tv di terima,infonya free mola tv tp bayar diskusi os nya tidak diupdate pembelian belum tv terima,info free mola tv tp bayar diskusi os nya tidak diupdate beli belum tv terima
7,8,Pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai syang nya gak saringanya mengkukus kali teliti lgi,pesan sesuai syang nya gak saringanya kukus kali teliti lgi
8,9,remote control tidak peka,remote control tidak peka,remote control tidak peka,remote control tidak peka,remote control tidak peka,remote control tidak peka
9,10,Sudah sampai mesin cuci nya terima kasih cepat ya 5 hari sampai...

Tabel 10 contoh disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\outputs\contoh_10.csv


## 7. Analisis Dampak Preprocessing

Bagian ini mengukur dampak secara kuantitatif:

- **Panjang teks**: rata-rata jumlah karakter dan kata sebelum (teks asli) vs sesudah (teks final).
- **Vocabulary**: jumlah kata unik sebelum (token hasil cleaning, sebelum stopword/stemming) vs sesudah (token final), beserta persentase penurunan.
- **Kata paling sering**: 10 kata teratas sebelum vs sesudah stopword removal.

In [10]:
df["panjang_asli_char"] = df[TEXT_COL].astype(str).str.len()
df["jumlah_kata_asli"] = df[TEXT_COL].astype(str).str.split().str.len()
df["panjang_final_char"] = df["teks_final"].str.len()
df["jumlah_kata_final"] = df["teks_final"].str.split().str.len()

avg_char_before = float(df["panjang_asli_char"].mean())
avg_char_after = float(df["panjang_final_char"].mean())
avg_word_before = float(df["jumlah_kata_asli"].mean())
avg_word_after = float(df["jumlah_kata_final"].mean())

ringkasan_panjang = pd.DataFrame({
    "Metrik": ["Rata-rata panjang karakter", "Rata-rata jumlah kata"],
    "Sebelum": [round(avg_char_before, 2), round(avg_word_before, 2)],
    "Sesudah": [round(avg_char_after, 2), round(avg_word_after, 2)],
})
ringkasan_panjang["Perubahan (%)"] = (
    (ringkasan_panjang["Sesudah"] - ringkasan_panjang["Sebelum"]) / ringkasan_panjang["Sebelum"] * 100
).round(2)
display(ringkasan_panjang)

vocab_before = set(token for tokens in df["tokenisasi"] for token in tokens)
vocab_after = set(token for tokens in df["stemming"] for token in tokens)
vocab_raw = set(
    token
    for tokens in df[TEXT_COL].astype(str).str.lower().str.split()
    for token in tokens
)
vocab_reduction = (len(vocab_before) - len(vocab_after)) / len(vocab_before) * 100

ringkasan_vocab = pd.DataFrame({
    "Tahap": ["Vocabulary mentah (lowercase split)", "Setelah cleaning (sebelum stopword/stemming)", "Setelah stopword removal + stemming"],
    "Jumlah kata unik": [len(vocab_raw), len(vocab_before), len(vocab_after)],
})
display(ringkasan_vocab)
print(f"Penurunan vocabulary dari {len(vocab_before)} menjadi {len(vocab_after)} kata unik: {vocab_reduction:.2f}%")

counter_before = Counter(token for tokens in df["tokenisasi"] for token in tokens)
counter_after = Counter(token for tokens in df["stopword_removal"] for token in tokens)
top_before = pd.DataFrame(counter_before.most_common(10), columns=["kata", "frekuensi"])
top_after = pd.DataFrame(counter_after.most_common(10), columns=["kata", "frekuensi"])

top_compare = pd.DataFrame({
    "Peringkat": range(1, 11),
    "Sebelum: kata": top_before["kata"],
    "Sebelum: frekuensi": top_before["frekuensi"],
    "Sesudah: kata": top_after["kata"],
    "Sesudah: frekuensi": top_after["frekuensi"],
})
display(top_compare)
top_before.to_csv(OUT_DIR / "top10_sebelum.csv", index=False)
top_after.to_csv(OUT_DIR / "top10_sesudah.csv", index=False)

metrics = {
    "jumlah_data": int(len(df)),
    "jumlah_per_split": {str(k): int(v) for k, v in split_counts.items()},
    "kolom_teks": TEXT_COL,
    "kolom_label": LABEL_COL,
    "label_map": {str(k): str(v) for k, v in LABEL_MAP.items()},
    "distribusi_label": dist.to_dict(orient="records"),
    "missing_total": n_missing,
    "duplikat": n_dup,
    "rata_panjang_karakter_sebelum": round(avg_char_before, 2),
    "rata_panjang_karakter_sesudah": round(avg_char_after, 2),
    "rata_jumlah_kata_sebelum": round(avg_word_before, 2),
    "rata_jumlah_kata_sesudah": round(avg_word_after, 2),
    "perubahan_panjang_karakter_persen": round((avg_char_after - avg_char_before) / avg_char_before * 100, 2),
    "perubahan_jumlah_kata_persen": round((avg_word_after - avg_word_before) / avg_word_before * 100, 2),
    "vocab_mentah": len(vocab_raw),
    "vocab_sebelum": len(vocab_before),
    "vocab_sesudah": len(vocab_after),
    "vocab_reduksi_persen": round(vocab_reduction, 2),
    "teks_kosong": n_empty,
    "teks_kosong_persen": round(n_empty / len(df) * 100, 2),
    "top10_sebelum": top_before.to_dict(orient="records"),
    "top10_sesudah": top_after.to_dict(orient="records"),
    "jumlah_stopword_sastrawi": len(sw_sastrawi),
    "jumlah_stopword_nltk": len(sw_nltk),
    "jumlah_stopword_dipakai": len(STOPWORDS),
    "negasi_dipertahankan": sorted(NEGATIONS & ALL_STOPWORDS),
    "jumlah_kata_unik_di_stem": len(stem_cache),
    "waktu_preprocessing_detik": round(elapsed, 2),
    "waktu_jalan": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, ensure_ascii=False, indent=2)
print("Metrik disimpan ke:", OUT_DIR / "metrics.json")

,Metrik,Sebelum,Sesudah,Perubahan (%)
0,Rata-rata panjang karakter,103.57,68.67,-33.7
1,Rata-rata jumlah kata,15.76,11.41,-27.6


,Tahap,Jumlah kata unik
0,Vocabulary mentah (lowercase split),23371
1,Setelah cleaning (sebelum stopword/stemming),12548
2,Setelah stopword removal + stemming,10066


Penurunan vocabulary dari 12548 menjadi 10066 kata unik: 19.78%


,Peringkat,Sebelum: kata,Sebelum: frekuensi,Sesudah: kata,Sesudah: frekuensi
0,1,barang,3838,barang,3838
1,2,bagus,3019,bagus,3019
2,3,di,2930,pengiriman,2824
3,4,pengiriman,2824,nya,2790
4,5,dan,2818,sesuai,1670
5,6,nya,2790,cepat,1657
6,7,dengan,2084,yg,1648
7,8,ada,1970,tidak,1433
8,9,baik,1847,harga,1277
9,10,lama,1675,ga,1159


Metrik disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\outputs\metrics.json


## 8. Kesimpulan

Preprocessing menurunkan noise dan menyeragamkan bentuk kata, tetapi juga memiliki risiko:

- **Pengurangan noise**: URL, angka, tanda baca, emoji, dan karakter khusus dibuang; spasi dirapikan. Akibatnya teks lebih bersih, tetapi isyarat non-verbal (emoji) dan informasi numerik (misalnya jumlah pembelian) ikut hilang.
- **Normalisasi variasi kata**: case folding menyatukan perbedaan kapitalisasi, sedangkan stemming menyatukan kata berimbuhan ke bentuk dasar. Variasi slang dan pengulangan huruf (mis. `bagusss`) tidak sepenuhnya tertangani.
- **Pengurangan dimensi vocabulary**: stopword removal dan stemming memangkas jumlah kata unik sehingga representasi teks menjadi lebih ringkas.
- **Risiko stopword removal**: daftar stopword standar umumnya memuat kata negasi seperti `tidak` dan `bukan`. Jika kata tersebut dibuang, makna kalimat dapat berbalik (mis. `tidak bagus` menjadi `bagus`). Pada notebook ini negasi dipertahankan secara eksplisit.
- **Risiko stemming**: stemmer dapat salah mereduksi kata (over-stemming/under-stemming), terutama pada kata tidak baku, singkatan, atau nama produk, sehingga makna asli dapat bergeser.

Secara keseluruhan, preprocessing membuat data lebih konsisten dan siap untuk analisis atau pemodelan sentimen, dengan catatan hasilnya perlu divalidasi pada kasus negasi dan kata tidak baku.

In [11]:
from IPython.display import Markdown, display as display_markdown

display_markdown(Markdown(f"""
**Ringkasan angka dari notebook ini**

- Data: **{len(df):,}** ulasan ({', '.join(f'{k}: {v:,}' for k, v in split_counts.items())}).
- Rata-rata panjang: **{avg_char_before:.2f}** karakter / **{avg_word_before:.2f}** kata → **{avg_char_after:.2f}** karakter / **{avg_word_after:.2f}** kata.
- Vocabulary: **{len(vocab_before):,}** → **{len(vocab_after):,}** kata unik (**turun {vocab_reduction:.2f}%**).
- Teks kosong setelah preprocessing: **{n_empty}** baris ({n_empty / len(df) * 100:.2f}%).
- Stopword dipakai: **{len(STOPWORDS)}** kata; negasi dipertahankan: **{', '.join(sorted(NEGATIONS & ALL_STOPWORDS))}**.
""".replace(",", ".")))


**Ringkasan angka dari notebook ini**

- Data: **11.606** ulasan (train: 11.606).
- Rata-rata panjang: **103.57** karakter / **15.76** kata → **68.67** karakter / **11.41** kata.
- Vocabulary: **12.548** → **10.066** kata unik (**turun 19.78%**).
- Teks kosong setelah preprocessing: **16** baris (0.14%).
- Stopword dipakai: **765** kata; negasi dipertahankan: **belum. bukan. enggak. jangan. kurang. nggak. tak. tanpa. tidak**.


## 9. Menyimpan Hasil

Hasil preprocessing disimpan ke `data/hasil_preprocessing.csv` (UTF-8). Kolom token disimpan sebagai string yang dipisahkan spasi agar berkas tetap berupa CSV yang valid. Ukuran berkas diperiksa; jika melebihi 50 MB, penyimpanan dibatalkan dan pengguna diberi peringatan.

In [12]:
save_columns = [
    "split_asal", TEXT_COL, LABEL_COL, "label_sentimen",
    "case_folding", "cleaning", "tokenisasi", "stopword_removal", "stemming", "teks_final", "teks_kosong",
]
out_df = df[save_columns].copy()
for col in ("tokenisasi", "stopword_removal", "stemming"):
    out_df[col] = out_df[col].apply(lambda tokens: " ".join(tokens))

out_path = DATA_DIR / "hasil_preprocessing.csv"
out_df.to_csv(out_path, index=False, encoding="utf-8")
size_mb = out_path.stat().st_size / (1024 * 1024)
print(f"Disimpan: {out_path}")
print(f"Jumlah baris: {len(out_df):,} | Jumlah kolom: {out_df.shape[1]} | Ukuran: {size_mb:.2f} MB")
if size_mb >= 50:
    print("PERINGATAN: ukuran melebihi 50 MB, pertimbangkan menyimpan versi terkompresi.")
else:
    print("Ukuran wajar (<50 MB).")

Disimpan: D:\NLP\preprocessing-ulasan-ecommerce-id\data\hasil_preprocessing.csv
Jumlah baris: 11,606 | Jumlah kolom: 11 | Ukuran: 7.22 MB
Ukuran wajar (<50 MB).
